# Análise do Desempenho Olímpico e Características Socioeconômicas

*Estudo exploratório sobre a evolução histórica das medalhas (1896–2024) e sua relação com indicadores demográficos e econômicos (1960–2024) nos Jogos Olímpicos de Verão.*

**Grupo:** Erick Souza Alberti, Felipe Augusto da Silva Cruz e Guilherme Henrique Pereira Cunha

**Repositório no GitHub:** [github.com/felipeascruz/AEDV](https://github.com/felipeascruz/AEDV)

## Introdução

Os Jogos Olímpicos constituem o principal palco esportivo internacional e carregam ampla relevância esportiva, sociocultural e de projeção institucional para as nações participantes. Ainda assim, resumir o desempenho das delegações apenas pela contagem bruta de medalhas dificulta comparações entre países com escalas demográficas, econômicas e estruturais profundamente distintas.

Este projeto investiga como o desempenho nos Jogos Olímpicos de Verão evoluiu historicamente e em que medida ele se associa a indicadores socioeconômicos externos. Ao longo do estudo, o desempenho é avaliado sob perspectivas complementares: contagem absoluta de medalhas, participação percentual no total de medalhas de cada edição, desvios em relação a países com perfil socioeconômico semelhante e distribuição entre provas individuais e por equipe.

Como as bases originais possuem coberturas temporais distintas (reunindo edições de Verão e de Inverno de 1896 a 2026), a análise é organizada em dois recortes temporais coerentes com a disponibilidade das fontes:

* **Recorte histórico olímpico (`1896–2024`, 31 edições):** utilizado nas análises estruturais de participação, distribuição e concentração de medalhas (Perguntas 1 e 2), além da análise desagregada por tipo de prova na edição de 2016 (Pergunta 8).
* **Recorte socioeconômico (`1960–2024`, 17 edições):** período de sobreposição com os indicadores anuais do World Bank — população total, Produto Interno Bruto (PIB) e PIB per capita (Perguntas 3 a 7).

### Roteiro das Questões Investigadas

1. **Evolução por edição (`1896–2024`):** Como o desempenho olímpico dos países, medido pelo número de medalhas conquistadas, varia entre as diferentes edições dos Jogos analisadas?
2. **Concentração histórica e por edição (`1896–2024`):** Como as medalhas estão distribuídas entre os países participantes e em que grau se concentram em um grupo restrito?
3. **Escala populacional (`1960–2024`):** Países com populações maiores tendem a conquistar mais medalhas?
4. **Escala econômica (`1960–2024`):** Existe associação entre o PIB total de um país e seu número de medalhas conquistadas?
5. **Economia agregada vs. renda por habitante (`1960–2024`):** A relação observada muda quando utilizamos PIB per capita em vez de PIB total?
6. **Desvios condicionais (`2024`):** Quais países apresentam desempenho olímpico muito acima ou abaixo do padrão observado para países com características socioeconômicas semelhantes?
7. **Estabilidade temporal (`1960–2024`):** Essas relações entre características socioeconômicas e desempenho olímpico permanecem semelhantes ao longo das diferentes edições dos Jogos?
8. **Perfis por tipo de prova (`2016`):** Os países que se destacam nas provas individuais também se destacam nas provas por equipe, ou existem perfis distintos de sucesso olímpico?

## Dados e Arquitetura do Projeto

O estudo integra três fontes de dados em uma base analítica estruturada na unidade **país × edição olímpica**:

| Fonte | Arquivo / Origem | Cobertura | Papel na Análise |
| :--- | :--- | :--- | :--- |
| **Quadro de Medalhas** | `data/medal_table.csv` (1.929 linhas) | 1896–2026 (Verão e Inverno) | Contagem oficial de ouro, prata, bronze e total por país e edição (esportes coletivos contam uma vez). |
| **Participação de Atletas** | `data/olympic_athletes.csv` (315.094 linhas) | 1896–2026 (Verão e Inverno) | Identificação de **todos** os países participantes (incluindo zero medalhas) e detalhamento por prova. |
| **Indicadores Socioeconômicos** | `data/world_bank.csv` (API World Bank) | 1960–2025 (anual) | População total (`SP.POP.TOTL`), PIB em US$ correntes (`NY.GDP.MKTP.CD`) e PIB per capita (`NY.GDP.PCAP.CD`). |

### Decisões de Integração e Modularização

* **Preservação de não medalhistas:** A lista de delegações participantes por edição é extraída da base de atletas e combinada à esquerda ao quadro de medalhas; participantes sem pódio recebem `0` em ouro, prata, bronze e total, evitando viés de seleção.
* **Agregação de subedições:** Em 1956, os registros de Melbourne e da subedição equestre de Estocolmo são somados por `year` e `noc` para obter uma única observação país × edição.
* **Padronização ISO $\rightarrow$ NOC:** Códigos ISO-3 do World Bank são convertidos explicitamente para NOCs olímpicos (excluindo `CHI` — *Channel Islands* — para evitar conflito com o Chile, unificando `ROC`/`OAR` em `RUS` e respeitando a transição do Líbano de `LIB` até 2016 para `LBN` a partir de 2020).
* **Tratamento de dados ausentes:** Delegações históricas sem correspondência defensável no World Bank (como União Soviética `URS`, Alemanha Oriental `GDR` e Equipe Unificada `EUN`) permanecem com `NaN` e não são imputadas; cada pergunta filtra apenas os indicadores exigidos para aquela análise.

> Para manter o notebook limpo e focado na leitura analítica, o pipeline de coleta, limpeza, padronização e auditoria foi unificado no módulo `src/dados.py`, enquanto a identidade visual ,inspirada nos princípios de Edward Tufte, com paleta Okabe-Ito, está centralizada em `estilo_grafico.py`.

In [ ]:
# Configuração global: imports centralizados e carregamento das bases integradas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.dados import carregar_bases, carregar_bases_brutas
from src.estilo_grafico import (
    COR_NEGATIVA,
    COR_PIB,
    COR_PIB_PER_CAPITA,
    COR_POPULACAO,
    COR_POSITIVA,
    COR_PRIMARIA,
    COR_SECUNDARIA,
    barras_agrupadas_com_valores,
    barras_com_valores,
    barras_horizontais_divergentes,
    cor_neutra,
    definir_marcas_y,
    eixo_log_em_potencias_de_dez,
    estilo_tufte,
    formatar_numero,
    legenda_sem_moldura,
    marcas_de_anos,
    marcas_do_eixo,
    moldura_de_intervalo,
    rotular_painel,
    rotular_ponto,
    rotulo_do_eixo,
    titulo_e_subtitulo,
)

df_olimpico_completo, df_socioeconomico = carregar_bases()

colunas_exibicao = [
    'year', 'noc', 'country', 'gold', 'silver', 'bronze', 'total',
    'population', 'gdp', 'gdp_per_capita',
]

print(f'Base histórica (1896–2024): {len(df_olimpico_completo):,} observações país × edição')
print(f'Base socioeconômica (1960–2024): {len(df_socioeconomico):,} observações país × edição')
display(df_socioeconomico[colunas_exibicao].head(5))

---
## I. Evolução e Concentração Histórica (1896–2024)

### 1. Como o desempenho olímpico dos países, medido pelo número de medalhas conquistadas, varia entre as diferentes edições dos Jogos analisadas?

**Metodologia e Recorte**
* Considera todas as 31 edições dos Jogos Olímpicos de Verão (`1896–2024`) na unidade país × edição.
* Mantém participantes sem pódio com `total = 0` para distinguir participação sem medalha de ausência na edição.
* Resume a distribuição por edição com contagem de participantes, número de países medalhistas, total de medalhas, média, mediana, terceiro quartil (`Q3`) e máximo por país.

In [ ]:
# Gráfico 1.1: Evolução de países participantes e países medalhistas
resumo_edicoes = (
    df_olimpico_completo
    .groupby('year')
    .agg(
        participantes=('noc', 'nunique'),
        medalhistas=('total', lambda x: (x > 0).sum()),
        total_medalhas=('total', 'sum'),
        media_medalhas=('total', 'mean'),
        mediana_medalhas=('total', 'median'),
        q3_medalhas=('total', lambda x: x.quantile(0.75)),
        max_medalhas=('total', 'max'),
    )
    .reset_index()
)

anos = resumo_edicoes['year']
primeira_edicao = resumo_edicoes.loc[anos.idxmin()]
ultima_edicao = resumo_edicoes.loc[anos.idxmax()]
valor_maximo = resumo_edicoes['participantes'].max()

series = [
    ('participantes', 'países participantes', COR_PRIMARIA),
    ('medalhistas', 'países medalhistas', COR_SECUNDARIA),
]

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(12, 6))

    for coluna, rotulo, cor in series:
        ax.plot(anos, resumo_edicoes[coluna], color=cor, linewidth=1.6, marker='o', markersize=4)
        rotular_ponto(
            ax,
            ultima_edicao['year'],
            ultima_edicao[coluna],
            f"{formatar_numero(ultima_edicao[coluna])} {rotulo}",
            cor,
        )

    # Ancoragem visual do ponto de partida em 1896
    rotular_ponto(
        ax,
        primeira_edicao['year'],
        primeira_edicao['participantes'],
        f"{int(primeira_edicao['participantes'])} participantes ({int(primeira_edicao['medalhistas'])} medalhistas)",
        COR_PRIMARIA,
        deslocamento=(6, 10),
    )

    ax.set_xticks(marcas_de_anos(anos))
    definir_marcas_y(ax, marcas_do_eixo(0, valor_maximo))
    moldura_de_intervalo(ax, anos.min(), anos.max(), 0, valor_maximo)
    titulo_e_subtitulo(
        ax,
        'Participação e conquista de medalhas nos Jogos Olímpicos de Verão',
        f'Número de países em cada edição, {anos.min()}–{anos.max()}',
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Gráfico 1.2: Evolução da média e da mediana de medalhas por país participante
anos = resumo_edicoes['year']
ultima_edicao = resumo_edicoes.loc[anos.idxmax()]
edicao_1904 = resumo_edicoes.loc[resumo_edicoes['year'] == 1904].iloc[0]
edicao_1964 = resumo_edicoes.loc[resumo_edicoes['year'] == 1964].iloc[0]
valor_maximo = resumo_edicoes['media_medalhas'].max()

series = [
    ('media_medalhas', 'média', COR_PRIMARIA),
    ('mediana_medalhas', 'mediana', COR_SECUNDARIA),
]

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(12, 6))

    for coluna, rotulo, cor in series:
        ax.plot(anos, resumo_edicoes[coluna], color=cor, linewidth=1.6, marker='o', markersize=4)
        rotular_ponto(
            ax,
            ultima_edicao['year'],
            ultima_edicao[coluna],
            f"{rotulo}: {formatar_numero(ultima_edicao[coluna], 1)}",
            cor,
        )

    # Anotações diretas nos pontos de inflexão históricos
    rotular_ponto(
        ax,
        edicao_1904['year'],
        edicao_1904['media_medalhas'],
        f"1904: pico de {formatar_numero(edicao_1904['media_medalhas'], 1)} (15 países)",
        COR_PRIMARIA,
        deslocamento=(8, -4),
    )
    rotular_ponto(
        ax,
        edicao_1964['year'],
        edicao_1964['mediana_medalhas'],
        'mediana = 0 a partir de 1964',
        COR_SECUNDARIA,
        deslocamento=(4, 10),
    )

    ax.set_xticks(marcas_de_anos(anos))
    definir_marcas_y(ax, marcas_do_eixo(0, valor_maximo))
    moldura_de_intervalo(ax, anos.min(), anos.max(), 0, valor_maximo)
    titulo_e_subtitulo(
        ax,
        'Medalhas por país participante ao longo das edições',
        f'Média e mediana do número de medalhas por país em cada edição, {anos.min()}–{anos.max()}',
    )

    fig.tight_layout()
    plt.show()

#### Análise

O desempenho olímpico dos países varia substancialmente entre as edições tanto em nível quanto em dispersão: a expansão dos Jogos multiplicou o volume total de medalhas, mas reduziu a mediana por país participante a zero.

* **Expansão da escala do evento (`1896` vs. `2024`):** O número de países participantes saltou de 12 para 207, enquanto o total de medalhas distribuídas cresceu de 120 para 1.044.
* **Crescimento assimétrico dos pódios:** Em 1896, 10 de 12 delegações (`83,3%`) conquistaram medalhas; em 2024, 92 de 207 delegações (`44,4%`) subiram ao pódio, deixando mais da metade dos participantes sem medalhas.
* **Queda da mediana para zero:** A média caiu de 10 medalhas por país em 1896 (com pico atípico de 18,6 em Saint Louis 1904, marcada por apenas 15 delegações) para 5,0 em 2024. Já a mediana recuou de 6 para 0 e permaneceu nula em todas as edições a partir de 1964.
* **Forte assimetria à direita:** Enquanto pelo menos metade das delegações encerra cada edição contemporânea com zero medalhas (e 75% com até 4 medalhas em 2024), potências isoladas ultrapassam a marca de 100 medalhas (máximo de 126 em 2024), puxando a média muito acima da mediana.

Essa distância crescente entre média e mediana evidencia uma distribuição altamente assimétrica, cuja concentração é investigada diretamente na questão seguinte.

### 2. Como as medalhas estão distribuídas entre os países participantes? Elas se concentram em um grupo relativamente pequeno de países?

**Metodologia e Recorte**
* Agrega o total de medalhas por código de Comitê Olímpico Nacional (NOC) em todas as edições de Verão (`1896–2024`), preservando delegações históricas (como `URS` e `GDR`) como entidades próprias.
* Ordena os NOCs do maior para o menor total histórico e calcula suas participações percentual e acumulada.
* Complementa a visão acumulada com a participação conjunta do Top 10 em cada edição, distinguindo concentração histórica de concentração competitiva contemporânea.

In [ ]:
# Gráfico 2.1: Curva de concentração histórica acumulada por NOC
df_q2 = df_olimpico_completo[df_olimpico_completo['noc'].notna()].copy()

medalhas_por_noc = (
    df_q2
    .groupby('noc', as_index=False)
    .agg(total_medalhas=('total', 'sum'))
    .sort_values('total_medalhas', ascending=False)
    .reset_index(drop=True)
)

total_medalhas_historico = medalhas_por_noc['total_medalhas'].sum()
medalhas_por_noc['participacao_pct'] = medalhas_por_noc['total_medalhas'] / total_medalhas_historico * 100
medalhas_por_noc['participacao_acumulada_pct'] = medalhas_por_noc['participacao_pct'].cumsum()
medalhas_por_noc['posicao'] = medalhas_por_noc.index + 1
posicao = medalhas_por_noc['posicao']
acumulado = medalhas_por_noc['participacao_acumulada_pct']
total_de_nocs = posicao.max()
referencias = [50, 80, 90]

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(posicao, acumulado, color=COR_PRIMARIA, linewidth=1.8)

    for referencia in referencias:
        ax.hlines(referencia, 1, total_de_nocs, color=cor_neutra(0.3), linewidth=0.8, zorder=0)
        primeiro = medalhas_por_noc[acumulado >= referencia].iloc[0]
        x, y = int(primeiro['posicao']), primeiro['participacao_acumulada_pct']
        ax.plot(x, y, 'o', color=COR_PRIMARIA, markersize=5)
        rotular_ponto(
            ax,
            x,
            y,
            f"{x} NOCs ({formatar_numero(y, 1)}%)",
            COR_PRIMARIA,
            deslocamento=(8, -12),
        )

    ax.set_xticks([1, *marcas_do_eixo(1, total_de_nocs, alvo=4), total_de_nocs])
    definir_marcas_y(ax, [0, *referencias, 100], sufixo='%')
    moldura_de_intervalo(ax, 1, total_de_nocs, 0, 100)
    rotulo_do_eixo(ax, 'NOCs ordenados pelo total de medalhas')
    titulo_e_subtitulo(
        ax,
        'Concentração histórica das medalhas olímpicas de Verão',
        'Percentual acumulado do total histórico de medalhas (1896–2024)',
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Gráfico 2.2: Participação conjunta dos 10 maiores medalhistas em cada edição
ranking_edicao = df_q2.sort_values(['year', 'total'], ascending=[True, False]).copy()
ranking_edicao['posicao_edicao'] = ranking_edicao.groupby('year').cumcount() + 1
ranking_edicao['total_edicao'] = ranking_edicao.groupby('year')['total'].transform('sum')
ranking_edicao['participacao_pct'] = ranking_edicao['total'] / ranking_edicao['total_edicao'] * 100

concentracao_top10 = (
    ranking_edicao[ranking_edicao['posicao_edicao'] <= 10]
    .groupby('year', as_index=False)
    .agg(participacao_top10=('participacao_pct', 'sum'))
)

anos = concentracao_top10['year']
participacao = concentracao_top10['participacao_top10']
primeira_edicao = concentracao_top10.loc[anos.idxmin()]
ultima_edicao = concentracao_top10.loc[anos.idxmax()]
minimo_edicao = concentracao_top10.loc[participacao.idxmin()]
minimo_y = int(participacao.min() // 10 * 10)

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(anos, participacao, color=COR_PRIMARIA, linewidth=1.6, marker='o', markersize=4)

    for edicao, deslocamento in (
        (primeira_edicao, (6, 12)),
        (minimo_edicao, (-18, -16)),
        (ultima_edicao, (8, 0)),
    ):
        rotular_ponto(
            ax,
            edicao['year'],
            edicao['participacao_top10'],
            f"{formatar_numero(edicao['participacao_top10'], 1)}% em {int(edicao['year'])}",
            COR_PRIMARIA,
            deslocamento=deslocamento,
        )

    ax.set_xticks(marcas_de_anos(anos))
    definir_marcas_y(ax, marcas_do_eixo(minimo_y, participacao.max()), sufixo='%')
    moldura_de_intervalo(ax, anos.min(), anos.max(), minimo_y, participacao.max())
    titulo_e_subtitulo(
        ax,
        'Participação dos 10 países com mais medalhas em cada edição',
        'Percentual das medalhas da edição conquistado pelo Top 10 (1896–2024)',
    )

    fig.tight_layout()
    plt.show()

#### Análise

As medalhas olímpicas estão fortemente concentradas em um grupo restrito de países, tanto no acumulado histórico (`1896–2024`) quanto dentro de cada edição contemporânea.

* **Patamares históricos (`1896–2024`):** Entre 233 NOCs participantes e 18.185 medalhas distribuídas, apenas 9 NOCs (`3,9%` do total) acumulam 50,1% de todas as medalhas (`USA` lidera isoladamente com 2.806 medalhas ou 15,43%, seguido por `GBR` com 5,60% e `URS` com 5,55%).
* **Concentração na cauda:** 20 NOCs reúnem 73,3% das conquistas, 26 NOCs atingem 80,2%, e 40 NOCs (`17,2%` das delegações) concentram 90,1% do total histórico. Dos 233 NOCs, 156 conquistaram ao menos uma medalha e 77 jamais subiram ao pódio.
* **Cautela na leitura acumulada vs. desempenho por edição:** O total histórico reflete também a longevidade de participação e a presença de delegações históricas extintas (`URS`, `GDR`). Ainda assim, a análise por edição confirma que a concentração permanece elevada: a fatia do Top 10 caiu de 100% em 1896 e 1904 (quando havia 10 e 7 medalhistas) para um piso de 53,3% em 2016, estabilizando-se entre 53% e 57% nas edições recentes (55,8% em 2024, quando 92 dos 207 NOCs conquistaram medalhas e o Top 20 concentrou 71,5%).

Essa persistência motiva investigar o papel de fatores estruturais de escala — começando pelo tamanho populacional e pelo PIB no recorte `1960–2024`.

---
## II. Escala Demográfica e Econômica (1960–2024)

### 3. Países com populações maiores tendem a conquistar mais medalhas?

**Metodologia e Recorte**
* Utiliza o recorte `1960–2024` (`2.679` observações país × edição), mantendo apenas registros com população disponível (`2.584` observações; `95` sem correspondência no World Bank, como delegações históricas).
* Adota escala logarítmica no eixo horizontal para acomodar diferenças de várias ordens de grandeza e calcula a correlação de Spearman, robusta à forte assimetria das variáveis.
* Agrupa os países em quartis de população dentro de cada edição (`Q1` a `Q4`), neutralizando o crescimento populacional global entre 1960 e 2024.

In [ ]:
# Gráfico 3.1: Dispersão entre população (escala log) e total de medalhas
df_q3 = df_socioeconomico.dropna(subset=['population']).copy()

populacao = df_q3['population']
medalhas = df_q3['total']
periodo = f"{df_q3['year'].min()}–{df_q3['year'].max()}"
correlacao_spearman_pop = (
    df_q3[['population', 'total']]
    .corr(method='spearman')
    .loc['population', 'total']
)

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(12, 7))

    ax.scatter(
        populacao,
        medalhas,
        s=16,
        facecolors='none',
        edgecolors=COR_POPULACAO,
        linewidths=0.8,
        alpha=0.7,
    )

    eixo_log_em_potencias_de_dez(ax, populacao)
    definir_marcas_y(ax, marcas_do_eixo(0, medalhas.max()))
    moldura_de_intervalo(ax, populacao.min(), populacao.max(), 0, medalhas.max())
    rotular_painel(ax, f"Correlação de Spearman = {formatar_numero(correlacao_spearman_pop, 2)}", COR_POPULACAO)
    rotulo_do_eixo(ax, 'População do país (escala logarítmica)')
    titulo_e_subtitulo(
        ax,
        f'População e número de medalhas nos Jogos Olímpicos de Verão ({periodo})',
        'Cada ponto é um país em uma edição; no eixo vertical, o total de medalhas na edição',
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Gráfico 3.2: Média de medalhas por quartil de população intra-edição
rotulos_quartis = [
    'Q1 - menor população',
    'Q2',
    'Q3',
    'Q4 - maior população',
]

df_q3['quartil_populacao'] = (
    df_q3
    .groupby('year')['population']
    .transform(lambda x: pd.qcut(x.rank(method='first'), q=4, labels=rotulos_quartis))
)

resumo_quartis_pop = (
    df_q3
    .groupby('quartil_populacao', observed=False)
    .agg(
        observacoes=('total', 'size'),
        media_medalhas=('total', 'mean'),
        mediana_medalhas=('total', 'median'),
        proporcao_medalhistas=('total', lambda x: (x > 0).mean() * 100),
    )
    .reset_index()
)

rotulos = resumo_quartis_pop['quartil_populacao'].astype(str).str.replace(' - ', '\n')

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(10, 6))

    barras_com_valores(ax, rotulos, resumo_quartis_pop['media_medalhas'], COR_POPULACAO)
    titulo_e_subtitulo(
        ax,
        'Média de medalhas por quartil de população',
        'Média de medalhas por país e edição, por quartil de população dentro de cada edição',
        recuo=0,
    )

    fig.tight_layout()
    plt.show()

#### Análise

Países com populações maiores tendem a conquistar mais medalhas, relação confirmada pela **correlação de postos de Spearman de $\rho = 0{,}477$. Essa métrica — adequada à forte assimetria dos dados e ao elevado volume de países sem medalhas — indica uma associação monotônica positiva moderada, mostrando que o tamanho populacional favorece o desempenho olímpico, mas a ampla variabilidade impede que o fator demográfico isolado determine o resultado.

- **Progressão monotônica entre quartis intra-edição:** Em linha com o coeficiente de Spearman, a média sobe de 0,14 medalha no quartil menos populoso (`Q1`) para 1,54 (`Q2`), 3,86 (`Q3`) e 12,28 medalhas no quartil mais populoso (`Q4`).
- **Probabilidade de pódio:** Apenas 9,3% das participações em `Q1` conquistaram ao menos uma medalha, contra 69,3% em `Q4` — único grupo com mediana positiva (3 medalhas, frente a 0 nos três primeiros quartis).
- **Dispersão e limite causal:** Nações muito populosas com menor estrutura esportiva frequentemente encerram edições sem pódios, enquanto países menos populosos alcançam resultados expressivos. Além disso, a associação observada não implica causalidade direta, motivando o exame da dimensão econômica nas questões seguintes.

### 4. Existe associação entre o PIB de um país e seu número de medalhas conquistadas?

**Metodologia e Recorte**
* Analisa as `2.447` observações país × edição (`1960–2024`) com PIB em dólares correntes (`NY.GDP.MKTP.CD`) disponível (`232` ausentes).
* Como valores nominais em dólares de 1960 e 2024 não são diretamente comparáveis devido à inflação, calcula a posição percentual (`percentil_gdp_edicao`) e os quartis de PIB dentro de cada edição para medir a associação principal por Spearman.
* Mantém o gráfico de dispersão em escala logarítmica como visualização exploratória da relação de ordem de grandeza.

In [ ]:
# Gráfico 4.1: Dispersão entre PIB em US$ correntes (escala log) e medalhas
df_q4 = df_socioeconomico.dropna(subset=['gdp']).copy()

df_q4['percentil_gdp_edicao'] = (
    df_q4
    .groupby('year')['gdp']
    .rank(method='average', pct=True)
)

correlacao_spearman_gdp = (
    df_q4[['percentil_gdp_edicao', 'total']]
    .corr(method='spearman')
    .loc['percentil_gdp_edicao', 'total']
)

pib = df_q4['gdp']
medalhas = df_q4['total']
periodo = f"{df_q4['year'].min()}–{df_q4['year'].max()}"

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(12, 7))

    ax.scatter(
        pib,
        medalhas,
        s=16,
        facecolors='none',
        edgecolors=COR_PIB,
        linewidths=0.8,
        alpha=0.7,
    )

    eixo_log_em_potencias_de_dez(ax, pib, prefixo='US$ ')
    definir_marcas_y(ax, marcas_do_eixo(0, medalhas.max()))
    moldura_de_intervalo(ax, pib.min(), pib.max(), 0, medalhas.max())
    rotular_painel(ax, f"Correlação de Spearman = {formatar_numero(correlacao_spearman_gdp, 2)}", COR_PIB)
    rotulo_do_eixo(ax, 'PIB em US$ correntes (escala logarítmica)')
    titulo_e_subtitulo(
        ax,
        f'PIB e número de medalhas nos Jogos Olímpicos de Verão ({periodo})',
        'Cada ponto é um país em uma edição; no eixo vertical, o total de medalhas na edição',
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Gráfico 4.2: Média de medalhas por quartil de PIB intra-edição
rotulos_quartis_gdp = [
    'Q1 - menor PIB',
    'Q2',
    'Q3',
    'Q4 - maior PIB',
]

df_q4['quartil_gdp'] = (
    df_q4
    .groupby('year')['gdp']
    .transform(lambda x: pd.qcut(x.rank(method='first'), q=4, labels=rotulos_quartis_gdp))
)

resumo_quartis_gdp = (
    df_q4
    .groupby('quartil_gdp', observed=False)
    .agg(
        observacoes=('total', 'size'),
        media_medalhas=('total', 'mean'),
        mediana_medalhas=('total', 'median'),
        proporcao_medalhistas=('total', lambda x: (x > 0).mean() * 100),
    )
    .reset_index()
)

rotulos = resumo_quartis_gdp['quartil_gdp'].astype(str).str.replace(' - ', '\n')

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(10, 6))

    barras_com_valores(ax, rotulos, resumo_quartis_gdp['media_medalhas'], COR_PIB)
    titulo_e_subtitulo(
        ax,
        'Média de medalhas por quartil de PIB',
        'Média de medalhas por país e edição, por quartil de PIB dentro de cada edição',
        recuo=0,
    )

    fig.tight_layout()
    plt.show()

#### Análise

Existe associação positiva clara entre o PIB total de um país e seu número de medalhas ($\rho = 0,67$), substancialmente mais forte do que a observada com a população ($\rho = 0,48$).

* **Forte gradiente econômico entre quartis:** A média salta de 0,11 medalha no quartil de menor PIB (`Q1`) para **0,58** (`Q2`), **3,10** (`Q3`) e 14,18 medalhas no quartil superior (`Q4`).
* **Alta taxa de conversão em pódio:** No topo econômico (`Q4`), 86,9% das delegações conquistaram ao menos uma medalha (com mediana de 5 medalhas), frente a apenas 7,3% em `Q1`.
* **Interpretação e cautelas:** Como o PIB total combina escala populacional e capacidade produtiva, ele captura simultaneamente a disponibilidade de recursos para financiar delegações amplas e infraestrutura esportiva — sem, contudo, informar a distribuição de renda nem comprovar causalidade direta.

### 5. A relação observada muda quando utilizamos PIB per capita em vez de PIB total?

**Metodologia e Recorte**
* Restringe a análise à amostra comum (`2.447` observações país × edição em `1960–2024`) com PIB total e PIB per capita disponíveis e estritamente positivos, garantindo comparação pareada.
* Calcula a posição percentual (`0%` a `100%`) e os quartis de ambos os indicadores dentro de cada edição, confrontando as correlações de Spearman e as médias por quartil lado a lado.

In [ ]:
# Gráfico 5.1: Dispersão comparativa por percentil intra-edição
indicadores_q5 = {'gdp': 'PIB total', 'gdp_per_capita': 'PIB per capita'}
df_q5 = (
    df_socioeconomico
    .dropna(subset=list(indicadores_q5))
    .query('gdp > 0 and gdp_per_capita > 0')
    .sort_values(['year', 'noc'])
    .copy()
)

correlacoes_q5 = {}
resumos_q5 = []

for coluna, nome in indicadores_q5.items():
    percentil = f'percentil_{coluna}_q5'
    quartil = f'quartil_{coluna}_q5'
    df_q5[percentil] = df_q5.groupby('year')[coluna].rank(method='average', pct=True)
    correlacoes_q5[nome] = df_q5[[percentil, 'total']].corr(method='spearman').iloc[0, 1]
    df_q5[quartil] = df_q5.groupby('year')[coluna].transform(
        lambda x: pd.qcut(x.rank(method='first'), 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    )
    resumo = (
        df_q5
        .groupby(quartil, observed=False)
        .agg(
            observacoes=('total', 'size'),
            media_medalhas=('total', 'mean'),
            mediana_medalhas=('total', 'median'),
            proporcao_medalhistas=('total', lambda x: (x > 0).mean() * 100),
        )
        .rename_axis('quartil')
        .reset_index()
    )
    resumo['indicador'] = nome
    resumos_q5.append(resumo)

resumo_quartis_q5 = pd.concat(resumos_q5, ignore_index=True)

cores_q5 = {'PIB total': COR_PIB, 'PIB per capita': COR_PIB_PER_CAPITA}
medalhas_maximo = df_q5['total'].max()
marcas_x = [0, 25, 50, 75, 100]

with estilo_tufte():
    fig, eixos = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

    for eixo, (coluna, nome) in zip(eixos, indicadores_q5.items()):
        eixo.scatter(
            df_q5[f'percentil_{coluna}_q5'] * 100,
            df_q5['total'],
            s=16,
            facecolors='none',
            edgecolors=cores_q5[nome],
            linewidths=0.8,
            alpha=0.7,
        )

        eixo.set_xticks(marcas_x)
        eixo.set_xticklabels([f'{marca}%' for marca in marcas_x])
        moldura_de_intervalo(eixo, 0, 100, 0, medalhas_maximo)
        rotular_painel(eixo, f"{nome} | Correlação de Spearman = {formatar_numero(correlacoes_q5[nome], 2)}", cores_q5[nome])
        rotulo_do_eixo(eixo, 'Posição percentual do indicador na edição')

    definir_marcas_y(eixos[0], marcas_do_eixo(0, medalhas_maximo))
    titulo_e_subtitulo(
        eixos[0],
        'PIB total e PIB per capita: associação com medalhas (1960–2024)',
        'Cada ponto é um país em uma edição; no eixo vertical, o total de medalhas na edição',
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Gráfico 5.2: Média de medalhas por quartil de PIB total vs. PIB per capita
tabela_medias_q5 = resumo_quartis_q5.pivot(index='quartil', columns='indicador', values='media_medalhas')
rotulos = tabela_medias_q5.index.astype(str)

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(10, 6))

    barras_agrupadas_com_valores(
        ax,
        rotulos,
        {
            'PIB total': (tabela_medias_q5['PIB total'], COR_PIB),
            'PIB per capita': (tabela_medias_q5['PIB per capita'], COR_PIB_PER_CAPITA),
        },
    )
    titulo_e_subtitulo(
        ax,
        'Média de medalhas por quartil de PIB total e PIB per capita',
        'Média de medalhas por participação, por quartil do indicador dentro de cada edição',
        recuo=0,
    )

    fig.tight_layout()
    plt.show()

#### Análise

Ao substituir o PIB total pelo PIB per capita na mesma amostra pareada (`2.447` observações), a associação permanece positiva, mas perde quase metade da intensidade ($\rho$ cai de `0,67` para `0,37`).

* **Menor diferenciação entre extremos:** Enquanto a média por quartil de PIB total vai de 0,11 (`Q1`) a 14,18 medalhas (`Q4`), no PIB per capita a variação é mais gradual: 0,76 (`Q1`), 2,36 (`Q2`), 4,29 (`Q3`) e 10,58 medalhas (`Q4`).
* **Pódio no topo da distribuição:** No quartil superior (`Q4`), 86,9% das observações de PIB total conquistam medalhas (mediana de 5), contra 61,8% no topo do PIB per capita (mediana de 2).
* **Por que isso ocorre:** Pequenos países de altíssima renda per capita (como Luxemburgo, Liechtenstein ou pequenos estados insulares) enfrentam limite estrutural de tamanho de delegação, enquanto grandes economias populosas conseguem disputar dezenas de modalidades simultaneamente.

---
## III. Desempenho Condicional, Estabilidade Temporal e Perfis Esportivos

### 6. Quais países apresentam desempenho olímpico muito acima ou abaixo do padrão observado para países com características socioeconômicas semelhantes?

**Metodologia e Recorte**
* Analisa um retrato transversal dos Jogos de Verão de `2024` (`191` países com população e PIB per capita válidos e positivos; exclui o registro isolado inconsistente convertido de `ROC` para `RUS` sem quadro de medalhas).
* Define semelhança socioeconômica pela distância euclidiana sobre o logaritmo natural padronizado (média `0`, desvio-padrão `1`) de população e PIB per capita — evitando a redundância de incluir simultaneamente o PIB total.
* Compara o total de medalhas de cada país com a mediana dos `k = 10` vizinhos mais próximos (verificando estabilidade com `k = 5` e `k = 15`).

In [ ]:
# Gráfico 6.1: Maiores desvios positivos e negativos em 2024 (k = 10 vizinhos)
ANO_Q6 = 2024
NUM_VIZINHOS_Q6 = 10

df_q6_edicao = df_socioeconomico[df_socioeconomico['year'] == ANO_Q6].copy()
df_q6 = (
    df_q6_edicao
    .dropna(subset=['population', 'gdp_per_capita'])
    .query('population > 0 and gdp_per_capita > 0 and noc != "RUS"')
    .sort_values('noc')
    .reset_index(drop=True)
)

indicadores_log_q6 = np.log(df_q6[['population', 'gdp_per_capita']])
indicadores_padronizados_q6 = (
    (indicadores_log_q6 - indicadores_log_q6.mean()) / indicadores_log_q6.std(ddof=0)
).to_numpy()

distancias_q6 = np.sqrt(
    ((indicadores_padronizados_q6[:, None, :] - indicadores_padronizados_q6[None, :, :]) ** 2).sum(axis=2)
)
np.fill_diagonal(distancias_q6, np.inf)
ordem_vizinhos_q6 = np.argsort(distancias_q6, axis=1, kind='stable')

for k in [5,10,15]:
    vizinhos = ordem_vizinhos_q6[:, :k]
    df_q6[f'referencia_{k}'] = np.median(df_q6['total'].to_numpy()[vizinhos], axis=1)
    df_q6[f'diferenca_{k}'] = df_q6['total'] - df_q6[f'referencia_{k}']

vizinhos_q6 = ordem_vizinhos_q6[:, :NUM_VIZINHOS_Q6]
df_q6['vizinhos'] = [', '.join(df_q6.iloc[indices]['noc']) for indices in vizinhos_q6]
df_q6['distancia_media'] = np.take_along_axis(distancias_q6, vizinhos_q6, axis=1).mean(axis=1)
df_q6['distancia_maxima'] = np.take_along_axis(distancias_q6, vizinhos_q6, axis=1).max(axis=1)

acima_q6 = df_q6[df_q6['diferenca_10'] > 0].sort_values(['diferenca_10', 'noc'], ascending=[False, True]).head(5)
abaixo_q6 = df_q6[df_q6['diferenca_10'] < 0].sort_values(['diferenca_10', 'noc'], ascending=[True, True]).head(5)
destaques_q6 = pd.concat([acima_q6, abaixo_q6])

grafico_q6 = destaques_q6.sort_values(['diferenca_10', 'noc'])

with estilo_tufte():
    fig, ax = plt.subplots(figsize=(12, 7))

    barras_horizontais_divergentes(
        ax,
        grafico_q6['country'] + ' (' + grafico_q6['noc'] + ')',
        grafico_q6['diferenca_10'],
        COR_POSITIVA,
        COR_NEGATIVA,
    )
    titulo_e_subtitulo(
        ax,
        f'Maiores desvios de medalhas em relação a países semelhantes — {ANO_Q6}',
        'Medalhas do país − mediana dos dez países mais próximos em população e PIB per capita',
        recuo=0,
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Verificação de sensibilidade ao número de vizinhos (k = 5, 10 e 15)
display(
    destaques_q6[['noc', 'country', 'diferenca_5', 'diferenca_10', 'diferenca_15']]
    .rename(columns={
        'diferenca_5': '5 vizinhos',
        'diferenca_10': '10 vizinhos (base)',
        'diferenca_15': '15 vizinhos',
    })
    .round(1)
)

#### Análise

Países com perfil socioeconômico próximo exibem diferenças acentuadas de medalhas em 2024 (`191` países comparados), evidenciando o peso de fatores institucionais, culturais e de investimento esportivo.

* **Maiores desvios positivos (`k = 10` vizinhos):** Estados Unidos (`+89,5` medalhas; `126` vs. mediana `36,5`), China (`+85`; `91` vs. `6`), Austrália (`+34`; `53` vs. `19`), Grã-Bretanha (`+32,5`; `65` vs. `32,5`) e França (`+31,5`; `64` vs. `32,5`) superam amplamente seus pares.
* **Maiores desvios negativos (`k = 10` vizinhos):** Arábia Saudita (`−32,5`; `0` vs. `32,5`), Espanha (`−18,5`; `18` vs. `36,5`), Emirados Árabes Unidos (`−8,0`; `0` vs. `8`), Canadá (`−6,5`; `27` vs. `33,5`) e Finlândia (`−6`; `0` vs. `6,0`) ficam abaixo do padrão de seus grupos — lembrando que o desvio mede distância relativa ao grupo de pares, não classificação absoluta (o Canadá conquistou `27` medalhas, mas abaixo da mediana de `33,5` de seus vizinhos).
* **Estabilidade e limitações metodológicas:** Todos os dez destaques preservam o sinal do desvio com `5`, `10` ou `15` vizinhos (embora a magnitude oscile, como no Canadá, que varia de `−26,0` para `−5,0`). Nos extremos de escala (`USA` e `CHN`), as distâncias médias são maiores (`0,93` e `1,19`), pois há poucos pares comparáveis; além disso, diferenças absolutas favorecem grandes totais positivos e representam um retrato descritivo de `2024`, não uma medida causal de eficiência.

### 7. Essas relações entre características socioeconômicas e desempenho olímpico permanecem semelhantes ao longo das diferentes edições dos Jogos?

**Metodologia e Recorte**
* Calcula, separadamente em cada uma das 17 edições (`1960–2024`), a correlação de Spearman entre total de medalhas e população, PIB total e PIB per capita na mesma amostra válida por ano.
* Evita comparações monetárias nominais entre décadas distintas ao estimar os coeficientes estritamente dentro de cada edição.
* Confronta a amostra completa disponível em cada edição (cuja cobertura sobe de `72,6%` em 1960 para `92,7%` em 2024) com um painel fixo de `25 NOCs` presentes em todas as 17 edições, isolando efeitos de composição da amostra.

In [ ]:
# Gráfico 7.1: Evolução das correlações de Spearman por edição (1960–2024)
indicadores_q7 = {'population': 'População', 'gdp': 'PIB total', 'gdp_per_capita': 'PIB per capita'}
base_q7 = df_socioeconomico.dropna(subset=['noc']).copy()
base_q7 = base_q7[~((base_q7['year'] == 2024) & (base_q7['noc'] == 'RUS'))].copy()

df_q7 = base_q7.dropna(subset=list(indicadores_q7)).copy()
df_q7 = df_q7[(df_q7[list(indicadores_q7)] > 0).all(axis=1)].copy()

cobertura_q7 = (
    base_q7.groupby('year').size().rename('participantes').to_frame()
    .join(df_q7.groupby('year').size().rename('amostra_valida'))
)
cobertura_q7['cobertura_pct'] = 100 * cobertura_q7['amostra_valida'] / cobertura_q7['participantes']

anos_q7 = sorted(base_q7['year'].unique())
frequencia_q7 = df_q7.groupby('noc')['year'].nunique()
nocs_fixos_q7 = frequencia_q7[frequencia_q7 == len(anos_q7)].index
df_q7_fixo = df_q7[df_q7['noc'].isin(nocs_fixos_q7)].copy()

resultados_q7 = []
for amostra, dados in [('Participantes com dados', df_q7), ('Painel fixo', df_q7_fixo)]:
    for ano, grupo in dados.groupby('year'):
        for coluna, nome in indicadores_q7.items():
            rho = grupo[[coluna, 'total']].corr(method='spearman').iloc[0, 1]
            resultados_q7.append({
                'year': ano,
                'indicador': nome,
                'amostra': amostra,
                'n': len(grupo),
                'spearman': rho,
            })

correlacoes_q7 = pd.DataFrame(resultados_q7)

cores_q7 = {'População': COR_POPULACAO, 'PIB total': COR_PIB, 'PIB per capita': COR_PIB_PER_CAPITA}
deslocamentos_q7 = {'PIB total': (8, 0), 'População': (8, 6), 'PIB per capita': (8, -6)}
principal_q7 = correlacoes_q7[correlacoes_q7['amostra'] == 'Participantes com dados']

with estilo_tufte():
    fig, eixo = plt.subplots(figsize=(13, 6))

    for nome, grupo in principal_q7.groupby('indicador', sort=False):
        eixo.plot(grupo['year'], grupo['spearman'], marker='o', markersize=4, linewidth=1.6, color=cores_q7[nome])
        ultima = grupo.loc[grupo['year'].idxmax()]
        rotular_ponto(
            eixo,
            ultima['year'],
            ultima['spearman'],
            f"{nome}: {formatar_numero(ultima['spearman'], 2)}",
            cores_q7[nome],
            deslocamento=deslocamentos_q7[nome],
        )

    eixo.hlines(0, min(anos_q7), max(anos_q7), color=cor_neutra(0.4), linewidth=0.8, zorder=0)
    eixo.set_xticks(anos_q7)
    definir_marcas_y(eixo, [0, 0.2, 0.4, 0.6, 0.8, 1])
    moldura_de_intervalo(eixo, min(anos_q7), max(anos_q7), 0, 1)
    titulo_e_subtitulo(
        eixo,
        'Associação entre indicadores socioeconômicos e medalhas por edição',
        'Correlação de Spearman entre cada indicador e o número de medalhas, em cada edição',
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Gráfico 7.2: Sensibilidade à composição dos países (amostra completa vs. painel fixo de 25 NOCs)
anos = correlacoes_q7['year']

with estilo_tufte():
    fig, eixos = plt.subplots(1, 3, figsize=(10.5, 5.6), sharex=True, sharey=True)

    for i, (eixo, nome) in enumerate(zip(eixos, indicadores_q7.values())):
        for amostra, estilo in [('Participantes com dados', '-'), ('Painel fixo', '--')]:
            dados = correlacoes_q7[
                (correlacoes_q7['indicador'] == nome) & (correlacoes_q7['amostra'] == amostra)
            ]
            cor = cores_q7[nome] if estilo == '-' else cor_neutra(0.7)
            eixo.plot(
                dados['year'],
                dados['spearman'],
                linestyle=estilo,
                marker='o',
                markersize=3.5,
                linewidth=1.5,
                color=cor,
            )

        eixo.set_xticks(marcas_de_anos(anos))
        eixo.set_ylim(-0.05, 1.03)
        moldura_de_intervalo(eixo, anos.min(), anos.max(), 0, 1)
        rotular_painel(eixo, nome, cores_q7[nome])

        if i > 0:
            eixo.spines['left'].set_visible(False)
            eixo.tick_params(axis='y', length=0)

    definir_marcas_y(eixos[0], [0, 0.2, 0.4, 0.6, 0.8, 1])
    legenda_sem_moldura(eixos[0], [('Participantes com dados', '-'), ('Painel fixo (25 NOCs)', '--')], loc='lower left')
    titulo_e_subtitulo(
        eixos[0],
        'Sensibilidade à composição dos países analisados',
        'Correlação de Spearman entre cada indicador e o número de medalhas, em cada edição',
    )

    fig.subplots_adjust(left=0.06, right=0.98, top=0.84, bottom=0.10, wspace=0.08)
    plt.show()

#### Análise

A hierarquia entre os indicadores permanece semelhante ao longo das 17 edições (`1960–2024`) — com o PIB total sempre na liderança —, mas a intensidade varia ao longo do tempo e depende da composição da amostra.

* **Hierarquia consistente na amostra completa:** Em todas as edições entre 1960 e 2024, as três associações são positivas: PIB total (`0,58` a `0,79`) supera população (`0,42` a `0,53`) e PIB per capita (`0,33` a `0,44`)** (com leve inversão entre população e PIB per capita apenas em 1976: `0,42` vs. `0,44`).
* **Estabilidade pós-Guerra Fria:** Após oscilações nos anos de boicote olímpico (`1980` registra o mínimo do PIB total em `0,58`), os coeficientes estabilizam-se de 1992 a 2024 em torno de **`0,66–0,69` para PIB total, `0,43–0,50` para população e `0,34–0,42` para PIB per capita.
* **Efeito de composição no painel fixo (`25 NOCs`):** Entre os países observados continuamente desde 1960, o PIB total segue elevado (`0,66` a `0,79`), mas a população sobe de `0,29` para `0,43` enquanto o PIB per capita perde força gradualmente (de `0,47` em 1960 para `0,12` em 2024, chegando a `−0,02` em 2016), mostrando que a entrada de novas nações em desenvolvimento ao longo das décadas explica parte da correlação transversal do PIB per capita.

### 8. Os países que se destacam nas provas individuais também se destacam nas provas por equipe, ou existem perfis distintos de sucesso olímpico?

**Metodologia e Recorte**
* Utiliza os Jogos Olímpicos de Verão de `2016` porque uma auditoria prévia revelou que, em 2020 e 2024, `74` provas (sobretudo duplas, revezamentos e modalidades coletivas) não possuem registros de medalhistas na base de atletas, enquanto 2016 possui cobertura de 100% (`306` de `306` provas).
* Classifica cada prova em **Individual** (`226` provas) ou **Por equipe** (`80` provas), validando a classificação pela multiplicidade máxima de atletas por conquista — e contabiliza cada medalha coletiva uma única vez por `NOC × prova × medalha × equipe`.
* Normaliza os resultados pela participação percentual de cada NOC no total mundial de cada categoria, corrigindo a maior oferta de medalhas individuais (`733`) frente às coletivas (`240`).

In [ ]:
# Gráfico 8.1: Participação nas medalhas individuais vs. por equipe (2016)
df_medalhas_bruto, df_atletas_bruto = carregar_bases_brutas()

linhas_auditoria_q8 = []
for ano in [2016, 2020, 2024]:
    dados = df_atletas_bruto[df_atletas_bruto['year'] == ano]
    eventos = dados[['sport', 'event']].drop_duplicates()
    premiados = dados[dados['medal'].isin(['Gold', 'Silver', 'Bronze'])]
    eventos_premiados = premiados[['sport', 'event']].drop_duplicates()
    linhas_auditoria_q8.append({
        'ano': ano,
        'provas_na_base': len(eventos),
        'provas_com_medalhistas': len(eventos_premiados),
        'provas_sem_medalhistas': len(eventos) - len(eventos_premiados),
    })

ANO_Q8 = 2016
atletas_q8 = df_atletas_bruto[df_atletas_bruto['year'] == ANO_Q8].copy()
classificacao_q8 = atletas_q8[['sport', 'event']].drop_duplicates().copy()

esportes_coletivos_q8 = [
    'Basketball', 'Football', 'Handball', 'Hockey',
    'Rugby Sevens', 'Volleyball', 'Water Polo', 'Synchronized Swimming',
]
padrao_equipe_q8 = (
    r'Team|Relay|Doubles|Fours|Pairs|Eights|Double Sculls|Quadruple|'
    r'Synchronized|Group|Two Person|Skiff|Multihull'
)
classificacao_q8['categoria'] = np.where(
    classificacao_q8['sport'].isin(esportes_coletivos_q8)
    | classificacao_q8['event'].str.contains(padrao_equipe_q8, case=False, regex=True),
    'Por equipe',
    'Individual',
)

medalhistas_q8 = (
    atletas_q8[atletas_q8['medal'].isin(['Gold', 'Silver', 'Bronze'])]
    .merge(classificacao_q8, on=['sport', 'event'], validate='many_to_one')
)

multiplicidade_q8 = (
    medalhistas_q8.groupby(['sport', 'event', 'noc', 'medal'])['id']
    .nunique()
    .groupby(['sport', 'event'])
    .max()
    .rename('max_atletas_por_conquista')
)
classificacao_q8 = classificacao_q8.merge(multiplicidade_q8, on=['sport', 'event'], validate='one_to_one')
assert (classificacao_q8.query('categoria == "Por equipe"')['max_atletas_por_conquista'] > 1).all()
assert (classificacao_q8.query('categoria == "Individual"')['max_atletas_por_conquista'] == 1).all()

individuais_q8 = medalhistas_q8.query('categoria == "Individual"').drop_duplicates(
    ['year', 'sport', 'event', 'noc', 'medal', 'id']
)
equipes_q8 = medalhistas_q8.query('categoria == "Por equipe"').drop_duplicates(
    ['year', 'sport', 'event', 'noc', 'medal', 'team']
)
conquistas_q8 = pd.concat([individuais_q8, equipes_q8], ignore_index=True)

# Reconciliação explícita com o quadro oficial agregado (KAZ e VEN)
contagem_q8 = conquistas_q8.groupby(['noc', 'medal']).size().unstack(fill_value=0)
quadro_q8 = (
    df_medalhas_bruto[df_medalhas_bruto['year'] == ANO_Q8]
    .groupby('noc')[['gold', 'silver', 'bronze']]
    .sum()
    .rename(columns={'gold': 'Gold', 'silver': 'Silver', 'bronze': 'Bronze'})
)
diferencas_q8 = contagem_q8.subtract(quadro_q8, fill_value=0)
divergencias_q8 = diferencas_q8[(diferencas_q8 != 0).any(axis=1)]

perfis_q8 = conquistas_q8.groupby(['noc', 'categoria']).size().unstack(fill_value=0)
for categoria in ['Individual', 'Por equipe']:
    perfis_q8[f'participacao_{categoria}'] = 100 * perfis_q8[categoria] / perfis_q8[categoria].sum()
perfis_q8['total'] = perfis_q8['Individual'] + perfis_q8['Por equipe']
perfis_q8['diferenca_pp'] = perfis_q8['participacao_Por equipe'] - perfis_q8['participacao_Individual']

nomes_q8 = df_medalhas_bruto[df_medalhas_bruto['year'] == ANO_Q8].drop_duplicates('noc').set_index('noc')['country']
perfis_q8['country'] = perfis_q8.index.to_series().map(nomes_q8).fillna(perfis_q8.index.to_series())
correlacao_q8 = perfis_q8[['Individual', 'Por equipe']].corr(method='spearman').iloc[0, 1]


mais_equipe_q8 = perfis_q8[perfis_q8['diferenca_pp'] > 0].nlargest(5, 'diferenca_pp')
mais_individual_q8 = perfis_q8[perfis_q8['diferenca_pp'] < 0].nsmallest(5, 'diferenca_pp')
destaques_q8 = pd.concat([mais_equipe_q8, mais_individual_q8])

rotulos_q8 = set(perfis_q8.nlargest(10, 'total').index)
limite_q8 = perfis_q8[['participacao_Individual', 'participacao_Por equipe']].max().max() * 1.12
marcas_q8 = marcas_do_eixo(0, limite_q8)

with estilo_tufte():
    fig, eixo = plt.subplots(figsize=(9, 8))

    eixo.scatter(
        perfis_q8['participacao_Individual'],
        perfis_q8['participacao_Por equipe'],
        s=40,
        facecolors='none',
        edgecolors=COR_POSITIVA,
        linewidths=1,
        alpha=0.8,
    )

    eixo.plot([0, limite_q8], [0, limite_q8], color=cor_neutra(0.5), linewidth=0.8, zorder=0)
    eixo.annotate(
        'participações iguais',
        xy=(limite_q8 * 0.6, limite_q8 * 0.6),
        xytext=(0, 6),
        textcoords='offset points',
        rotation=45,
        rotation_mode='anchor',
        ha='center',
        va='bottom',
        fontsize=10,
        color=cor_neutra(0.7),
    )

    for noc in sorted(rotulos_q8):
        r = perfis_q8.loc[noc]
        eixo.annotate(
            noc,
            (r['participacao_Individual'], r['participacao_Por equipe']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=9,
        )

    eixo.set(xlim=(-0.4, limite_q8), ylim=(-0.4, limite_q8))
    eixo.set_aspect('equal', adjustable='box')
    eixo.set_xticks(marcas_q8)
    eixo.set_xticklabels([f'{marca}%' for marca in marcas_q8])
    definir_marcas_y(eixo, marcas_q8, sufixo='%')
    moldura_de_intervalo(eixo, 0, limite_q8, 0, limite_q8)
    rotulo_do_eixo(eixo, 'Participação nas medalhas individuais (%)')
    titulo_e_subtitulo(
        eixo,
        'Perfis de sucesso olímpico: provas individuais e por equipe — 2016',
        'Eixo vertical: participação nas medalhas por equipe (%)',
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# Gráfico 8.2: Maiores diferenças relativas entre provas por equipe e individuais (2016)
grafico_q8 = destaques_q8.sort_values('diferenca_pp')
sem_divergencias_q8 = perfis_q8.drop(index=divergencias_q8.index, errors='ignore')
rho_q8 = sem_divergencias_q8[['Individual', 'Por equipe']].corr(method='spearman').iloc[0, 1]

with estilo_tufte():
    fig, eixo = plt.subplots(figsize=(11, 6))

    barras_horizontais_divergentes(
        eixo,
        grafico_q8.index,
        grafico_q8['diferenca_pp'],
        COR_POSITIVA,
        COR_NEGATIVA,
        casas=2,
    )
    titulo_e_subtitulo(
        eixo,
        'Maiores diferenças entre os perfis de medalhas — 2016',
        'Participação por equipe − participação individual (pontos percentuais)',
        recuo=0,
    )
    eixo.text(
        0.98,
        0.04,
        f'Correlação de Spearman (Individual × Equipe): ρ = {formatar_numero(rho_q8, 3)}',
        transform=eixo.transAxes,
        ha='right',
        va='bottom',
        fontsize=10,
        color=cor_neutra(0.75),
    )

    fig.tight_layout()
    plt.show()

#### Análise

Existe correspondência positiva moderada entre o desempenho nas duas categorias em 2016 ($\rho = 0,49$, ou `0,51` sem os dois NOCs com divergência entre fontes), mas coexistem perfis nacionais fortemente especializados.

* **Cobertura e reconciliação em 2016:** A auditoria prévia confirmou cobertura de `100%` das `306` provas em 2016 (contra `74` provas sem medalhistas na base de atletas em 2020 e 2024), resultando em 226 provas individuais (`733` medalhas) e 80 por equipe (`240` medalhas) — totalizando 973 conquistas entre 86 NOCs medalhistas (com pequenas divergências documentadas frente ao quadro agregado de `972` medalhas em `KAZ` e `VEN`).
* **Potências equilibradas no topo:** Os Estados Unidos lideram ambas as frentes com participações próximas (12,69% nas individuais e 11,67% nas por equipe; `93` e `28` medalhas), acompanhados por China (`7,37%` vs. `6,67%`) e Grã-Bretanha (`6,28%` vs. `8,75%`).
* **Perfis orientados a provas coletivas:** A Alemanha conquistou `21` medalhas em cada categoria, mas isso equivale a 8,75% das medalhas por equipe contra 2,86% das individuais (`+5,89 p.p.`), seguida por Austrália (`+4,89 p.p.`), Grã-Bretanha (`+2,47 p.p.`), Dinamarca (`+1,83 p.p.`) e Sérvia (`+1,67 p.p.`).
* **Perfis concentrados em provas individuais:** Entre os `86` NOCs medalhistas de 2016, 42 venceram exclusivamente em provas individuais — com destaque para Azerbaijão (`18` individuais, `0` por equipe; `−2,46 p.p.`), Cazaquistão (`18` vs. `0`), Quênia (`13` vs. `0`), Uzbequistão (`13` vs. `0`) e Cuba (`11` vs. `0`) — enquanto apenas 4 (`Áustria`, `Estônia`, `Fiji` e `Nigéria`) subiram ao pódio somente por equipe.

---
## Conclusão

A trajetória dos Jogos Olímpicos de Verão evidencia que a ampla expansão global do evento não desconcentrou o acesso ao pódio, consolidando uma hierarquia fortemente condicionada pela escala econômica e modulada por estratégias esportivas nacionais. Embora o porte da economia estabeleça o patamar competitivo de base, o desempenho efetivo de cada delegação depende da capacidade de converter recursos em estrutura de alto rendimento e de explorar especializações por tipo de disputa.

1. **Universalização com concentração persistente (`1896–2024`):** Embora o número de delegações tenha crescido de `12` para `207` e o volume de medalhas de `120` para `1.044`, a mediana por participante caiu de `6` para zero a partir de `1964`. Historicamente, apenas `9` dos `233` NOCs concentram `50,1%` de todas as medalhas e `40` NOCs acumulam `90,1%`, padrão que se mantém nas edições contemporâneas, nas quais o Top 10 retém entre `53%` e `57%` dos pódios.
2. **Primazia da escala econômica sobre a demográfica e a renda por habitante (`1960–2024`):** As três variáveis apresentam associação positiva e estável ao longo das 17 edições, mas o PIB total (`ρ = 0,67`) mostra relação substancialmente mais forte com o total de medalhas do que a população (`ρ = 0,48`) e o PIB per capita (`ρ = 0,37`), refletindo a importância combinada de escala populacional e capacidade financeira para sustentar delegações competitivas.
3. **Heterogeneidade condicional e especialização por modalidade:** Mesmo entre países com população e PIB per capita semelhantes em `2024`, observam-se desvios expressivos em relação aos pares (positivos em potências como `USA`, `CHN`, `AUS`, `GBR` e `FRA`; negativos em `KSA`, `ESP`, `UAE`, `CAN` e `FIN`). Além disso, a análise desagregada de `2016` mostra que o sucesso em provas individuais (`ρ = 0,49`) não garante destaque proporcional em provas por equipe, revelando especializações nacionais distintas.

### Limitações

- **Natureza observacional:** As correlações de Spearman e comparações por vizinhos próximos identificam associações descritivas, não efeitos causais diretos.
- **Cobertura histórica e de provas:** Delegações históricas sem série socioeconômica equivalente no World Bank (`URS`, `GDR`, `EUN`) permanecem ausentes nas análises de `1960–2024`, e lacunas nos registros de medalhistas por prova em `2020` e `2024` restringiram a comparação individual vs. equipe a `2016`.